# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbottabad123/flyrank-ml-track/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [11]:
# Convert fields that should be numeric
numeric_candidates = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "word_count"
]

for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Basic numeric distributions
available_numeric = [
    c for c in numeric_candidates
    if c in df.columns
]

print("Numeric distribution summary:")
display(df[available_numeric].describe().T)

# Missing-value check
print("\nMissing values:")
display(df[available_numeric].isna().sum().to_frame("missing"))

# Staleness distribution
if "days_since_last_update" in df.columns:
    stale_summary = pd.cut(
        df["days_since_last_update"],
        bins=[-1, 30, 90, 180, 365, float("inf")],
        labels=["0-30", "31-90", "91-180", "181-365", "365+"],
        include_lowest=True
    ).value_counts(sort=False)

    print("\nStaleness buckets:")
    display(stale_summary.to_frame("n"))

# Impression distribution
if "impressions_90d" in df.columns:
    print("\nImpression quantiles:")
    display(
        df["impressions_90d"]
        .quantile([0, .25, .50, .75, .90, .95, 1])
        .to_frame("impressions_90d")
    )

Numeric distribution summary:


,count,mean,std,min,25%,50%,75%,max
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
word_count,22301.0,3107.760325,1452.382598,8.0,2413.0,2877.00,3666.00,9546.0



Missing values:


,missing
days_since_last_update,0
impressions_90d,0
ctr,0
word_count,7699



Staleness buckets:


,n
days_since_last_update,
0-30,20480
31-90,175
91-180,9171
181-365,169
365+,5



Impression quantiles:


,impressions_90d
0.00,1.00
0.25,81.00
0.50,731.00
0.75,3615.25
0.90,12136.40
0.95,22996.50
1.00,517715.00


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [13]:
# ============================================================
# SECTION 2 — THREE SIGNAL TESTS
# No is_declining_label is used
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Make sure numeric columns are numeric
# ------------------------------------------------------------

if "days_since_last_update" in df.columns:
    df["days_since_last_update"] = pd.to_numeric(
        df["days_since_last_update"],
        errors="coerce"
    )

if "impressions_90d" in df.columns:
    df["impressions_90d"] = pd.to_numeric(
        df["impressions_90d"],
        errors="coerce"
    )

if "ctr" in df.columns:
    df["ctr"] = pd.to_numeric(
        df["ctr"],
        errors="coerce"
    )


# ============================================================
# TEST 1 — STALENESS
# Staleness vs median impressions
# ============================================================

print("=" * 60)
print("SIGNAL TEST #1 — STALENESS VS IMPRESSIONS")
print("=" * 60)

stale_bins = [-1, 30, 90, 180, 365, float("inf")]
stale_labels = [
    "0-30",
    "31-90",
    "91-180",
    "181-365",
    "365+"
]

df["stale_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=stale_bins,
    labels=stale_labels,
    include_lowest=True
)

t1 = (
    df.groupby(
        "stale_bucket",
        observed=True
    )["impressions_90d"]
    .agg(
        n="count",
        median_impressions="median"
    )
    .round(2)
)

display(t1)

# Automatic verdict
valid_t1 = t1.dropna(subset=["median_impressions"])

if len(valid_t1) < 2:
    verdict_1 = "FALSE"

elif valid_t1["median_impressions"].is_monotonic_decreasing:
    verdict_1 = "CONFIRMED"

elif valid_t1["median_impressions"].is_monotonic_increasing:
    verdict_1 = "OPPOSITE"

else:
    # Compare youngest and oldest buckets
    first_value = valid_t1["median_impressions"].iloc[0]
    last_value = valid_t1["median_impressions"].iloc[-1]

    if last_value < first_value:
        verdict_1 = "MIXED"
    elif last_value > first_value:
        verdict_1 = "OPPOSITE"
    else:
        verdict_1 = "FALSE"

print("\nVerdict #1:", verdict_1)


# ============================================================
# TEST 2 — WORD COUNT
# Word count tier vs median impressions
# ============================================================

print("\n")
print("=" * 60)
print("SIGNAL TEST #2 — WORD COUNT VS IMPRESSIONS")
print("=" * 60)

if "word_count_tier" in df.columns:

    t2 = (
        df.groupby(
            "word_count_tier",
            observed=True
        )["impressions_90d"]
        .agg(
            n="count",
            median_impressions="median"
        )
        .sort_values("median_impressions")
        .round(2)
    )

    display(t2)

    valid_t2 = t2.dropna(subset=["median_impressions"])

    if len(valid_t2) < 2:
        verdict_2 = "FALSE"

    elif valid_t2["median_impressions"].is_monotonic_increasing:
        verdict_2 = "CONFIRMED"

    elif valid_t2["median_impressions"].is_monotonic_decreasing:
        verdict_2 = "OPPOSITE"

    else:
        verdict_2 = "MIXED"

    print("\nVerdict #2:", verdict_2)

else:
    print("Column 'word_count_tier' not found.")
    verdict_2 = "FALSE"
    print("Verdict #2:", verdict_2)


# ============================================================
# TEST 3 — POSITION
# Position tier vs CTR
# Volume floor >= 500
# ============================================================

print("\n")
print("=" * 60)
print("SIGNAL TEST #3 — POSITION VS CTR")
print("=" * 60)

tier_order = [
    "top_3",
    "page_1",
    "striking",
    "page_3_5",
    "deep"
]

visible = df["impressions_90d"] >= 500

t3 = (
    df.loc[
        visible &
        df["position_tier"].isin(tier_order)
    ]
    .groupby(
        "position_tier",
        observed=True
    )["ctr"]
    .agg(
        n="count",
        median_ctr="median"
    )
    .reindex(tier_order)
    .round(4)
)

display(t3)

valid_t3 = t3.dropna(subset=["median_ctr"])

if len(valid_t3) < 2:
    verdict_3 = "FALSE"

elif valid_t3["median_ctr"].is_monotonic_decreasing:
    verdict_3 = "CONFIRMED"

elif valid_t3["median_ctr"].is_monotonic_increasing:
    verdict_3 = "OPPOSITE"

else:
    verdict_3 = "MIXED"

print("\nVerdict #3:", verdict_3)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("SIGNAL TEST SUMMARY")
print("=" * 60)

signal_summary = pd.DataFrame({
    "signal": [
        "Staleness vs impressions",
        "Word count vs impressions",
        "Position vs CTR"
    ],
    "verdict": [
        verdict_1,
        verdict_2,
        verdict_3
    ]
})

display(signal_summary)

SIGNAL TEST #1 — STALENESS VS IMPRESSIONS


,n,median_impressions
stale_bucket,,
0-30,20480,470.0
31-90,175,510.0
91-180,9171,1692.0
181-365,169,16.0
365+,5,2.0



Verdict #1: MIXED


SIGNAL TEST #2 — WORD COUNT VS IMPRESSIONS


,n,median_impressions
word_count_tier,,
<1000,973,4.0
1000-2000,3780,172.0
2000-3500,11263,997.0
3500+,6285,1340.0



Verdict #2: CONFIRMED


SIGNAL TEST #3 — POSITION VS CTR


,n,median_ctr
position_tier,,
top_3,458,0.20
page_1,7064,0.24
striking,4485,0.17
page_3_5,4330,0.09
deep,389,0.00



Verdict #3: MIXED


SIGNAL TEST SUMMARY


,signal,verdict
0,Staleness vs impressions,MIXED
1,Word count vs impressions,CONFIRMED
2,Position vs CTR,MIXED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [14]:
# ============================================================
# FLAG-LINKED TEST — STALENESS / REFRESH
# ============================================================

print("=" * 60)
print("FLAG-LINKED TEST — STALENESS / REFRESH")
print("=" * 60)

flag_test = (
    df.groupby(
        "stale_bucket",
        observed=True
    )["impressions_90d"]
    .agg(
        n="count",
        median_impressions="median"
    )
    .round(2)
)

display(flag_test)

valid_flag = flag_test.dropna(
    subset=["median_impressions"]
)

if len(valid_flag) < 2:

    flag_verdict = "FALSE"

else:

    youngest = valid_flag["median_impressions"].iloc[0]
    oldest = valid_flag["median_impressions"].iloc[-1]

    print("Youngest bucket median impressions:", youngest)
    print("Oldest bucket median impressions:", oldest)

    if oldest < youngest:
        flag_verdict = "CONFIRMED"

    elif oldest > youngest:
        flag_verdict = "OPPOSITE"

    else:
        flag_verdict = "MIXED"


print("\nFlag-linked verdict:", flag_verdict)

FLAG-LINKED TEST — STALENESS / REFRESH


,n,median_impressions
stale_bucket,,
0-30,20480,470.0
31-90,175,510.0
91-180,9171,1692.0
181-365,169,16.0
365+,5,2.0


Youngest bucket median impressions: 470.0
Oldest bucket median impressions: 2.0

Flag-linked verdict: CONFIRMED


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [15]:
print("Practice takeaway:")
print(
    "Use staleness, search position, and visibility signals as decision-support "
    "for prioritizing content reviews, not as proof of causation."
)
print(
    "Pages with mixed or opposite signals should be reviewed manually before "
    "taking action."
)

Practice takeaway:
Use staleness, search position, and visibility signals as decision-support for prioritizing content reviews, not as proof of causation.
Pages with mixed or opposite signals should be reviewed manually before taking action.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.